# 03 — Validasi hasil, visualisasi, dan load PostgreSQL

Jalankan notebook ini setelah `02_pyspark_transform.ipynb`. Fokusnya adalah membuktikan hasil ETL konsisten, membuat grafik untuk laporan, dan menyediakan load database yang aman serta opsional.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from pyspark.sql import functions as F
from src.config import get_paths
from src.spark_pipeline import create_spark

load_dotenv(PROJECT_ROOT / '.env')
paths = get_paths('retail_contaminated_dataset')
paths.charts_dir.mkdir(parents=True, exist_ok=True)
spark = create_spark('local[*]')

## 1. Validasi data Parquet
Pemeriksaan ini menjadi bukti bahwa hasil ETL dapat dibaca ulang, tidak kehilangan ID penting, dan tetap mempunyai nilai bisnis yang masuk akal.

In [ ]:
fact = spark.read.parquet(str(paths.parquet_dir))
validation = fact.agg(
    F.count('*').alias('rows'),
    F.countDistinct('receipt_id').alias('receipts'),
    F.sum(F.when(F.col('store_name').isNull(), 1).otherwise(0)).alias('missing_store'),
    F.sum(F.when(F.col('sku_name').isNull(), 1).otherwise(0)).alias('missing_sku'),
    F.sum(F.when(F.col('loyalty_segment').isNull(), 1).otherwise(0)).alias('missing_customer'),
    F.round(F.sum('total_value'), 2).alias('sales_value'),
    F.round(F.sum('gross_profit'), 2).alias('gross_profit'),
).first().asDict()
assert validation['missing_store'] == 0
assert validation['missing_sku'] == 0
assert validation['missing_customer'] == 0
validation

## 2. Membuat grafik laporan
Hanya hasil agregasi kecil yang dikonversi ke Pandas. Data transaksi penuh tetap diproses oleh Spark.

In [ ]:
monthly = (fact.groupBy('month').agg(F.sum('total_value').alias('sales_value'))
           .orderBy('month').toPandas())
stores = (fact.groupBy('store_name').agg(F.sum('total_value').alias('sales_value'))
          .orderBy(F.desc('sales_value')).limit(10).toPandas())
categories = (fact.groupBy('category').agg(F.sum('total_value').alias('sales_value'))
              .orderBy(F.desc('sales_value')).toPandas())

monthly['month'] = pd.to_datetime(monthly['month'])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['month'], monthly['sales_value'], color='#2563eb', linewidth=2)
ax.set(title='Tren Nilai Penjualan Bulanan', xlabel='Bulan', ylabel='Nilai penjualan')
ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(paths.charts_dir / 'monthly_sales.png', dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].barh(stores['store_name'][::-1], stores['sales_value'][::-1], color='#0f766e')
axes[0].set_title('10 Toko dengan Penjualan Tertinggi')
axes[0].set_xlabel('Nilai penjualan')
axes[1].bar(categories['category'], categories['sales_value'], color='#ea580c')
axes[1].set_title('Penjualan per Kategori')
axes[1].set_ylabel('Nilai penjualan')
axes[1].tick_params(axis='x', rotation=35)
fig.tight_layout()
fig.savefig(paths.charts_dir / 'store_and_category_sales.png', dpi=160)
plt.show()

## 3. Analisis risiko inventori
File flag dari pembuat dataset digunakan sebagai label referensi. Analisis ini menghitung banyaknya pasangan SKU–toko per jenis risiko.

In [ ]:
inventory = spark.read.option('header', True).option('inferSchema', True).csv(
    str(paths.source_dir / 'inventory_snapshot.csv')
)
flags = spark.read.option('header', True).option('inferSchema', True).csv(
    str(paths.source_dir / 'sku_inventory_flags.csv')
)
print(f'Baris snapshot inventori: {inventory.count():,}')
flags.groupBy('flag').count().orderBy(F.desc('count')).show(truncate=False)

## 4. Load PostgreSQL — opsional
Salin `.env.example` menjadi `.env`, isi koneksi PostgreSQL, lalu ubah `ENABLE_POSTGRES_LOAD=1`. Yang dimuat secara default hanya tabel agregasi sehingga proses ringan. Jangan menaruh kata sandi di notebook atau Git.

In [ ]:
if os.getenv('ENABLE_POSTGRES_LOAD', '0') == '1':
    from sqlalchemy import URL, create_engine, text

    url = URL.create(
        'postgresql+psycopg2',
        username=os.environ['POSTGRES_USER'],
        password=os.environ['POSTGRES_PASSWORD'],
        host=os.getenv('POSTGRES_HOST', 'localhost'),
        port=int(os.getenv('POSTGRES_PORT', '5432')),
        database=os.environ['POSTGRES_DB'],
    )
    engine = create_engine(url)
    with engine.begin() as conn:
        conn.execute(text('CREATE SCHEMA IF NOT EXISTS retail_analytics'))
    for folder in paths.validation_dir.iterdir():
        if folder.is_dir():
            table = spark.read.option('header', True).option('inferSchema', True).csv(str(folder))
            table.toPandas().to_sql(folder.name, engine, schema='retail_analytics',
                                  if_exists='replace', index=False)
    print('Tabel agregasi berhasil dimuat ke PostgreSQL.')
else:
    print('Load PostgreSQL dilewati. Atur ENABLE_POSTGRES_LOAD=1 jika database sudah siap.')

## 5. Benchmark untuk laporan
Jalankan dari terminal proyek: `python scripts/benchmark.py`. Untuk memasukkan ukuran penuh, gunakan `python scripts/benchmark.py --include-full`. Salin waktu proses yang dihasilkan ke tabel pada `reports/laporan_akhir.md`.

In [ ]:
spark.stop()